In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
HOME = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Database"
TARGET_PATH = r"\NBA\Stats 2000 to 2024\Regular Season.xlsx"
FEATURE_PATH = r"\Working db\Features\working_df_27-4-25.csv"

In [9]:
df1 = pd.read_excel(HOME + TARGET_PATH)
features =  pd.read_csv(HOME+FEATURE_PATH)

KeyboardInterrupt: 

# Add the column Year to the target

In [ ]:
DRAFT_2000 = HOME + r"\NBA\Draft Picks 2000 to 2024\NBA Draft Picks 2000.csv"
draft = pd.read_csv(DRAFT_2000)

In [ ]:
def add_years_played(nba_data, draft_data):
    """
    Add 'yrs' column to nba_data indicating years played, where:
    - Players drafted BEFORE 2000 start at yrs=2 in 2000 season
    - Players drafted IN 2000 start at yrs=1 in their rookie season
    - Players drafted AFTER 2000 follow normal counting (yrs=1 in first season)
    """
    # Create a set of players drafted in 2000
    drafted_2000 = set(draft_data['Player'].unique())
    
    # Find all players and their first seasons
    player_first_season = nba_data.groupby('Player')['season'].min().to_dict()
    
    # Adjust first season for pre-2000 players
    for player, first_season in player_first_season.items():
        if player not in drafted_2000 and first_season == 1:
            # This player started before 2000 (since their first season is 2000-01)
            player_first_season[player] = 0  # Will make 2000 season count as yrs=2
    
    # Calculate years played
    nba_data['yrs'] = nba_data.apply(
        lambda row: row['season'] - player_first_season.get(row['Player'], row['season']) + 1,
        axis=1
    )
    
    return nba_data
    
target = add_years_played(df1, draft)

# Merging the target with the features to work on it

I will merge NBA Data (the target = WS, WS/48, Win, MP, GP) with the features that I have in the other database.

In [6]:
target = df1[['Player', 'Team', 'Pos', 'G', 'MP', 
        'WS', 'WS/48', 'WS_x', 'WS/48_x', 'season', 'yrs']]

dic = {
    'WS' : 'WS_x',
    'WS/48' : 'WS/48_x'
}

for i, j in dic.items():
    target[i] = target[i].fillna(target[j])

target = target.drop(['WS/48_x', 'WS_x'], axis=1)
target.isna().sum()/len(target)

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_38804\312846313.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target[i] = target[i].fillna(target[j])


Player    0.000000
Team      0.000000
Pos       0.000000
G         0.000000
MP        0.000000
WS        0.000000
WS/48     0.000341
season    0.000000
yrs       0.000000
dtype: float64

In [ ]:
temps_actuel = 
print(temps_actuel)

2025-04-08 17:28:26.493364


In [19]:
timestamp = time.time()
print(timestamp)

1744126043.2990518


## Transpose of the df based on id

If two teams, we delete the two teams and we keep the 2TM (same for 3 teams).

In [91]:
# First, let's create a unique identifier by combining name and first season
target['first_season'] = target.groupby('Player')['season'].transform('min')

# Create a player_id that combines name and first season
target['player_id'] = target['Player'].str.lower().str.strip(
                    ).str.replace(' ', '').str.replace("'", '').str.replace('.', ''
                    )+ '_' + target['first_season'].astype(str)

In [92]:
multi_team_markers = ['2TM', '3TM', '4TM', '5TM'] #Remove row that contain the average of several teams
wide_df = target[~target['Team'].isin(multi_team_markers) #Pivot table
            ].pivot_table(index='player_id', columns='yrs', 
            values=['WS', 'WS/48', 'MP', 'G'], aggfunc='mean') 
wide_df.columns = [f'{stat}-{season}' for stat, season in wide_df.columns]
wide_df = wide_df.reset_index()

In [145]:
# First, let's create a unique identifier by combining name and first season
features['draft_season'] = features.groupby(['Player', 'Team'])['season'].transform('max') + 1 - 2000

# Create a player_id that combines name and first season
features['player_id'] = features['Player'].str.lower().str.strip(
                    ).str.replace(' ', '').str.replace("'", '').str.replace('.', ''
                    )+ '_' + features['draft_season'].astype(str).str[:-2]

In [149]:
# Get the index of the last season for each player
last_season_idx = features.groupby('player_id')['season'].idxmax()

# Filter the original dataframe using these indices
last_seasons = features.loc[last_season_idx]

In [150]:
test = pd.merge(wide_df, last_seasons, on='player_id')

## Merging

We have duplicate in the rows because they are duplicate in the Player's names. Therefore we will use Draft Picks data to merge on base of college, NBA team and player's name.

In [49]:
#Concatenate the draft picks data
for year in range(2005, 2025):
    DRAFT = HOME + rf"\NBA\Draft Picks 2000 to 2024\NBA Draft Picks {year}.csv"
    if year == 2005:
        draft = pd.read_csv(DRAFT_2000)
        draft['season'] = year
    else:
        draft = pd.concat([draft, pd.read_csv(DRAFT)])
        draft['season'] =  draft['season'].fillna(year)
    

In [51]:
draft

,Pk,Tm,Player,College,Yrs,G,MP,PTS,TRB,AST,...,FT%,MP.1,PTS.1,TRB.1,AST.1,WS,WS/48,BPM,VORP,season
0,1,NJN,Kenyon Martin,Cincinnati,15.0,757.0,23134.0,9325.0,5159.0,1439.0,...,0.629,30.6,12.3,6.8,1.9,48.0,0.100,0.1,12.1,2005.0
1,2,VAN,Stromile Swift,LSU,9.0,547.0,10804.0,4582.0,2535.0,275.0,...,0.699,19.8,8.4,4.6,0.5,21.3,0.095,-1.6,1.1,2005.0
2,3,LAC,Darius Miles,NaN,7.0,446.0,11730.0,4507.0,2190.0,840.0,...,0.590,26.3,10.1,4.9,1.9,9.5,0.039,-1.0,3.0,2005.0
3,4,CHI,Marcus Fizer,Iowa State,6.0,289.0,6032.0,2782.0,1340.0,352.0,...,0.691,20.9,9.6,4.6,1.2,2.7,0.022,-3.7,-2.6,2005.0
4,5,ORL,Mike Miller,Florida,17.0,1032.0,27812.0,10973.0,4376.0,2666.0,...,0.769,26.9,10.6,4.2,2.6,60.7,0.105,0.8,19.8,2005.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,56,PHO,Kevin McCullar Jr.,Kansas,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.0
56,57,MEM,Ulrich Chomche,NaN,1.0,7.0,32.0,5.0,8.0,2.0,...,0.500,4.6,0.7,1.1,0.3,0.0,-0.009,-7.3,0.0,2024.0
57,58,DAL,Ariel Hukporti,NaN,1.0,22.0,165.0,34.0,42.0,8.0,...,0.545,7.5,1.5,1.9,0.4,0.0,0.006,-5.6,-0.1,2024.0
58,59,PHI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024.0
